# Zadanie 3
#### Wczytanie danych

In [52]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
car_evaluation = fetch_ucirepo(id=19) 
  
# data (as pandas dataframes) 
X = car_evaluation.data.features 
y = car_evaluation.data.targets 
  
# metadata 
print(car_evaluation.metadata) 
  
# variable information 
print(car_evaluation.variables) 



{'uci_id': 19, 'name': 'Car Evaluation', 'repository_url': 'https://archive.ics.uci.edu/dataset/19/car+evaluation', 'data_url': 'https://archive.ics.uci.edu/static/public/19/data.csv', 'abstract': 'Derived from simple hierarchical decision model, this database may be useful for testing constructive induction and structure discovery methods.', 'area': 'Other', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1728, 'num_features': 6, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1988, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5JP48', 'creators': ['Marko Bohanec'], 'intro_paper': {'ID': 249, 'type': 'NATIVE', 'title': 'Knowledge acquisition and explanation for multi-attribute decision making', 'authors': 'M. Bohanec, V. Rajkovič', 'venue': '8th Intl Workshop on Expert Systems and their Applications, 

# 1. Pie chart
Na wykresie tortowym pokaż ile spośród wszystkich samochodów to samochody nieakcepto-
walne, akceptowalne, dobre i bardzo dobre

In [53]:
import pandas as pd
import plotly.express as px
car_eval_whole = pd.concat([X,y],axis=1)
nowe_nazwy = {
    "unacc" : "unacceptable", "acc" : "acceptable", "vgood" : "very good"
}
car_eval_whole["class"]=car_eval_whole["class"].replace(nowe_nazwy)
fig = px.pie(car_eval_whole,names='class',title = "Evaluated cars")
fig.update_traces(textinfo = "value+percent")
fig.update_layout(width=1000,height=600)
fig.show()

# 2. Wykres słupkowy
Na wykresie słupkowym pokaż ile jest samochodów 2, 3 i 4 drzwiowych w różnych klasach
bezpieczeństwa


In [54]:
import polars as pl
import plotly.graph_objects as go
cars_eval = pl.from_pandas(car_eval_whole)
two_door = cars_eval.filter(pl.col("doors")=="2").group_by("safety").agg(pl.len())
three_door = cars_eval.filter(pl.col("doors")=="3").group_by("safety").agg(pl.len())
four_door = cars_eval.filter(pl.col("doors")=="4").group_by("safety").agg(pl.len())

fig = go.Figure()

fig.add_trace(go.Bar(x=two_door["safety"],y=two_door["len"],name="2 doors"))
fig.add_trace(go.Bar(x=three_door["safety"],y=three_door["len"],name="3 doors"))
fig.add_trace(go.Bar(x=four_door["safety"],y=four_door["len"],name="4 doors"))
fig.update_layout(
    title = "Safety levels vs door counts",
    xaxis_title = "Safety",
    yaxis_title="Number of cars",
    xaxis = dict(
        categoryorder='array',
        categoryarray=['high','med','low']
    )
)


# 3. Radar plot
Dowiedź się czym jest wykres radarowy (ang. radar plot), nazywany też gwiazdowym lub
pajęczynowym (ang. star plot lub spider plot), i zrób go dla 5 wybranych atrybutów danych.

In [57]:
df_radar = car_eval_whole.copy()

df_radar["buying"] = df_radar["buying"].map({"low": 1, "med": 2, "high": 3, "vhigh": 4})
df_radar["maint"] = df_radar["maint"].map({"low": 1, "med": 2, "high": 3, "vhigh": 4})
df_radar["doors"] = df_radar["doors"].map({"2": 2, "3": 3, "4": 4, "5more": 5})
df_radar["lug_boot"] = df_radar["lug_boot"].map({"small": 1, "med": 2, "big": 3})
df_radar["safety"] = df_radar["safety"].map({"low": 1, "med": 2, "high": 3})

df_grouped = df_radar.groupby("class")[
    ["buying", "maint", "doors", "lug_boot", "safety"]
].mean().reset_index()

df_melted = df_grouped.melt(
    id_vars="class", 
    var_name="attribute", 
    value_name="mean"
)

fig = px.line_polar(
    df_melted,r = "mean",theta="attribute",line_close=True,title="Radar plot for evaluated cars",color="class"
)
fig.update_traces(fill='toself',opacity=0.5)
fig.show()


# 4. Reguła wyboru dobrych i bardzo dobrych samochodów
 Spróbuj określić jakie wartości jakich atrybutów decydują o tym, że samochód jest dobry
lub bardzo dobry. Zrób to intuicyjnie, według własnego pomysłu - w dalszej części wykładu
nauczysz się to robić metodami uczenia maszynowego. Sformułuj regułę, która na podstawie
wartości kilku wybranych atrybutów mówi TAK (dobry lub bardzo dobry) lub NIE. Czy Twoja
reguła działa prawidłowo dla wszystkich danych czy tylko dla części, i jak dużej części? Zrób
rysunek (według własnego pomysłu) pokazujący gdzie reguła robi błęd

In [63]:
# safety = high , maint - low/med - lug_boot - med/big, doors >=3

car_eval_whole['actual_YES']= car_eval_whole['class'].isin(['good','very good'])

regula = (
    (car_eval_whole['safety'] == 'high') & ((car_eval_whole['maint'].isin(['med'])) & (car_eval_whole['lug_boot'].isin(['med', 'big'])) & (car_eval_whole['doors'].isin(['3', '4', '5more'])))
)
car_eval_whole['predicted_YES'] = regula

car_eval_whole['correct'] = (car_eval_whole['actual_YES'] == car_eval_whole['predicted_YES'])
accuracy = car_eval_whole['correct'].mean() * 100

print(f"Accuracy of this rule: {accuracy:.2f}%")

error = car_eval_whole.groupby(['persons','correct']).size().reset_index(name='count')
error['Wynik'] = error['correct'].map({True : "Correct", False : "Incorrect"})

fig = px.bar(
    error,
    x='persons',
    y='count',
    color = 'Wynik',
    title = "Error of intuitive rule to determine car's evaluation by persons capacity",
    barmode='group',
    color_discrete_sequence=['#ef553b', '#00cc96']
)
fig.show()

Accuracy of this rule: 90.74%


### Zasada działa dla 90.74% samochodów, co przekłada się na 160 błędów na 1728 samochodów